In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

df = pd.read_csv('/content/bq-results-20260323-203006-1774297820102.csv')
print(f"Shape: {df.shape}, Readmission rate: {df['readmitted_30d'].mean():.4f}")



In [3]:
# Missing values
df['marital_status']     = df['marital_status'].fillna('UNKNOWN')
df['language']           = df['language'].fillna('UNKNOWN')
df['insurance']          = df['insurance'].fillna('UNKNOWN')
df['admission_location'] = df['admission_location'].fillna('UNKNOWN')
df['discharge_location'] = df['discharge_location'].fillna('UNKNOWN')
df['race']               = df['race'].fillna('UNKNOWN')

lab_cols = ['num_lab_tests_24h', 'num_abnormal_labs', 'hemoglobin_min',
            'wbc_max', 'creatinine_max', 'sodium_min', 'sodium_max',
            'potassium_min', 'potassium_max', 'glucose_min', 'glucose_max']
for col in lab_cols:
    df[col] = df[col].fillna(df[col].median())

hist_cols = ['days_since_last_discharge', 'num_admissions_last_30d',
             'num_admissions_last_90d', 'num_admissions_last_year',
             'total_prior_admissions', 'recent_admission_flag', 'frequent_flyer_flag']
for col in hist_cols:
    df[col] = df[col].fillna(0)

df = df.fillna(df.median(numeric_only=True))



In [4]:
# Label encoding
categorical_cols = ['gender', 'race', 'marital_status', 'language', 'insurance',
                    'admission_location', 'discharge_location', 'admission_type']

le_mappings = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_mappings[col] = {
        'classes':      list(le.classes_),
        'int_to_label': dict(enumerate(le.classes_)),
        'label_to_int': {v: k for k, v in enumerate(le.classes_)}
    }

# Features
drop_cols = ['subject_id', 'hadm_id', 'admittime', 'dischtime',
             'readmitted_30d', 'readmitted_60d', 'readmitted_90d',
             'days_to_next_admission']
feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"Feature count: {len(feature_cols)}")



Feature count: 45


In [5]:
# Chronological sort and 70/15/15 split
df['admittime'] = pd.to_datetime(df['admittime'])
df_sorted = df.sort_values('admittime').reset_index(drop=True)

X = df_sorted[feature_cols]
y = df_sorted['readmitted_30d']

n         = len(X)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

X_train, y_train = X.iloc[:train_end],        y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],          y.iloc[val_end:]

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# Chronological integrity check
train_max = df_sorted.iloc[:train_end]['admittime'].max()
test_min  = df_sorted.iloc[val_end:]['admittime'].min()
assert test_min > train_max, "SPLIT FAILURE"
print(f"Chronological check passed: train ends {train_max}, test starts {test_min}")



Train: 284,221 | Val: 60,905 | Test: 60,905
Chronological check passed: train ends 2171-02-15 00:00:00, test starts 2183-02-15 12:15:00


In [6]:
# Train
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    subsample=0.8, reg_lambda=2, reg_alpha=0.1,
    n_estimators=500, min_child_weight=5, max_depth=5,
    learning_rate=0.05, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42, eval_metric='auc', verbosity=0, tree_method='hist'
)
xgb_model.fit(X_train, y_train)

auroc_train = roc_auc_score(y_train, xgb_model.predict_proba(X_train)[:, 1])
auroc_val   = roc_auc_score(y_val,   xgb_model.predict_proba(X_val)[:, 1])
auroc_test  = roc_auc_score(y_test,  xgb_model.predict_proba(X_test)[:, 1])
print(f"Train AUROC: {auroc_train:.4f} | Val AUROC: {auroc_val:.4f} | Test AUROC: {auroc_test:.4f}")

# Save
pickle.dump(xgb_model,   open('/content/xgboost_final_45f.pkl', 'wb'))
pickle.dump(feature_cols, open('/content/final_feature_cols.pkl', 'wb'))
pickle.dump(le_mappings,  open('/content/final_le_mappings.pkl', 'wb'))
pickle.dump({'train_end': train_end, 'val_end': val_end, 'n_total': n,
             'train_max_date': str(train_max), 'test_min_date': str(test_min)},
            open('/content/final_split_indices.pkl', 'wb'))

print("Saved: xgboost_final_45f.pkl, final_feature_cols.pkl, final_le_mappings.pkl, final_split_indices.pkl")

Train AUROC: 0.7535 | Val AUROC: 0.7238 | Test AUROC: 0.7183
Saved: xgboost_final_45f.pkl, final_feature_cols.pkl, final_le_mappings.pkl, final_split_indices.pkl
